# Proteomics analysis

**Goal**: get peptides whose expression levels are significantly varying within groups

### Imports and config loading

In [ ]:
import os
import sys
import copy
from pathlib import Path
import yaml
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import probplot, shapiro, f_oneway, kruskal, gaussian_kde
from statsmodels.stats.multitest import multipletests

pd.set_option('future.no_silent_downcasting', True)


class DotDict(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__


def _to_dotdict(x):
    return DotDict({k: _to_dotdict(v) if isinstance(v, dict) else v for k, v in x.items()})


def _detect_project_root():
    env_root = os.environ.get("LSMS_DATA_NORMALIZATION_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "params").is_dir() and (candidate / "scripts").is_dir():
            return candidate

    candidates = []
    try:
        nb_path = Path(__file__).resolve()
        candidates.extend([nb_path.parent, *nb_path.parents])
    except NameError:
        pass
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    visited = set()
    for candidate in candidates:
        if not candidate:
            continue
        resolved = candidate.expanduser().resolve()
        if resolved in visited:
            continue
        visited.add(resolved)
        if (resolved / "params").is_dir() and (resolved / "scripts").is_dir():
            return resolved

    raise RuntimeError("Could not locate project root. Set LSMS_DATA_NORMALIZATION_ROOT.")


PROJECT_ROOT = _detect_project_root()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")


def _load_params(params_id: str):
    candidates = sorted(Path("params").glob(f"{params_id}_*.yml"))
    if not candidates:
        raise FileNotFoundError(f"No params YAML found for ID {params_id} in ./params")
    with open(candidates[0], "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def _deep_merge(base: dict, extra: dict):
    for key, value in extra.items():
        if (
            key in base
            and isinstance(base[key], dict)
            and isinstance(value, dict)
        ):
            _deep_merge(base[key], value)
        else:
            base[key] = copy.deepcopy(value)
    return base


FILE_PATH_INPUT = os.getenv("FILE_PATH_INPUT", "./data")
FILE_PATH_OUTPUT = os.getenv("FILE_PATH_OUTPUT", "./results")
PROTEOMICS_PARAMS_ID = os.getenv("PROTEOMICS_PARAMS_ID", "003")

params_prot = _load_params(PROTEOMICS_PARAMS_ID)

cfg = DotDict(
    {
        "FILE_PATH_INPUT": FILE_PATH_INPUT,
        "FILE_PATH_OUTPUT": FILE_PATH_OUTPUT,
        **_to_dotdict(params_prot),
    }
)


Loading file to dataframe

In [ ]:
with pd.ExcelFile(os.path.join(FILE_PATH_INPUT, cfg.data.file_name_proteomics_input)) as xls:
    sheet_names = xls.sheet_names
    final_sheets = [name for name in sheet_names if name.upper().startswith("FINAL")]

    if not final_sheets:
        raise ValueError(f"No sheet starting with 'FINAL' found in {cfg.data.file_name_proteomics_input}. Available sheets: {sheet_names}")
    
    df = pd.read_excel(xls, sheet_name=final_sheets[0])
    print(f"Loaded sheet '{final_sheets[0]}' from {cfg.data.file_name_proteomics_input}")


### Step 1: Dataframe cleaning and preview

In [ ]:
# Fixing double-row header
df = df[1:]
# Forward fill missing values (aka fixing merged cells)
df.ffill(inplace=True)
print(f"Successfully performed forward fill on missing values.")

sample_prefix_arr = ("BIOPD", "RBD", "CON")
value_cols = [c for c in df.columns if c.startswith(sample_prefix_arr)]

groups = {
    "PD": [c for c in value_cols if c.startswith("BIOPD")],
    "RBD": [c for c in value_cols if c.startswith("RBD")],
    "CON": [c for c in value_cols if c.startswith("CON")]
}

print(f"Step 1: Preview of the first 15 rows of the dataframe after forward fill:")
print(df.head(15))


### Step 2: Removing samples with more than specified threshold of zero values (<LOQ; default: 60%)

In [ ]:
zero_threshold = cfg.preprocessing.zero_threshold if cfg.preprocessing.zero_threshold is not None else 0.6

df[value_cols] = df[value_cols].replace("<LOQ", np.nan).astype(float)

loq_fraction = df[value_cols].isna().mean(axis=1)

df_filtered = df.loc[loq_fraction <= zero_threshold].reset_index(drop=True)
print(f"Step 2: Filtered peptides that have more than %d%% of '<LOQ': kept %d of %d" % (zero_threshold * 100, len(df_filtered), len(df)))
df = df_filtered.copy()


### Step 3: Imputing remaining '<LOQ' values with LOQ/2 for each peptide

In [ ]:
for col in value_cols:
    min_detected = df[col].min(skipna=True)
    impute_value = min_detected / 2 if pd.notna(min_detected) else 0
    df.fillna({col: impute_value}, inplace=True)

print(f"Step 3: Imputed remaining '<LOQ' values with LOQ/2 for each peptide.")


### Step 4: Shapiro-Wilk test for normality on each peptide across samples

In [ ]:
def do_shapiro(df, groups):
    results = []
    for _, row in df.iterrows():
        for group, cols in groups.items():
            values = row[cols].astype(float).values
            if len(values) < 3:
                continue

            stat, pval = shapiro(values)
            _, (slope, intercept, r) = probplot(values)
            
            results.append({
                "peptide": row["Peptide"],
                "protein": row["Protein"],
                "group": group,

                "is_normal_shapiro": pval > 0.05,
                "is_normal_qq": r >= 0.990,

                "shapiro_stat": stat,
                "shapiro_pval": pval,

                "qq_slope": slope,
                "qq_intercept": intercept,
                "qq_rvalue": r
            })
    return results

# * Applied Shapiro-Wilk test on INITIAL peptide data (before log transformation)
# ? normality_df: peptide, protein, group, is_normal_shapiro, is_normal_qq, 
# ?               shapiro_stat, shapiro_pval, qq_slope, qq_intercept, qq_rvalue
normality_df = pd.DataFrame(do_shapiro(df, groups))
normality_df.sort_values(by=["peptide"], inplace=True) # by=["group", "peptide"]

# * Count percentage of non-normal peptides
# ? normal: array of peptides and their groups that have is_normal_shapiro == True
normal = normality_df.loc[normality_df["is_normal_shapiro"], ["peptide", "group"]]
print(f"Number of normal group-peptides: %d out of %d (%.2f%%)" % (normal.shape[0], normality_df.shape[0], normal.shape[0] / normality_df.shape[0] * 100))

# * Peptides normal in all 3 groups (by Shapiro)
normal_all_3 = (
    normality_df.groupby("peptide")
    .agg(n_groups=("group", "nunique"), all_normal=("is_normal_shapiro", "all"))
    .query("n_groups == 3 and all_normal")
    .index.tolist()
)
print(f"Peptides normal in all 3 groups (Shapiro): %d" % len(normal_all_3))


# * Apply log10 transformation and re-test normality
df_log = df.copy()
df_log[value_cols] = np.log10(
    df[value_cols].astype(float).clip(lower=1e-12)
)
normality_df_log = pd.DataFrame(do_shapiro(df_log, groups))
normality_df_log.sort_values(by=["peptide", "group"], inplace=True)

# * Count percentage of non-normal peptides after log transformation
normal_log = normality_df_log.loc[normality_df_log["is_normal_shapiro"], ["peptide", "group"]]
print(f"Number of normal group-peptides (after log transformation): %d out of %d (%.2f%%)" % (normal_log.shape[0], normality_df_log.shape[0], normal_log.shape[0] / normality_df_log.shape[0] * 100))

# * Peptides normal in all 3 groups (by Shapiro)
normal_all_3 = (
    normality_df_log.groupby("peptide")
    .agg(n_groups=("group", "nunique"), all_normal=("is_normal_shapiro", "all"))
    .query("n_groups == 3 and all_normal")
    .index.tolist()
)
print(f"Peptides normal in all 3 groups (Shapiro): %d" % len(normal_all_3))

print(f"Step 4: Completed Shapiro-Wilk tests for each peptide within each group.")


### Step 5: Doing statistical tests between groups

In [ ]:
# TODO: condition for choosing test based on normality results
# * Using Kruskal-Wallis test for all peptides (non-parametric)
kw_results = []
group_names = list(groups.keys())
for _, row in df.iterrows():
    values_by_group = [row[groups[g]].astype(float).values for g in group_names]
    stat, pval = kruskal(*values_by_group)
    kw_results.append({
        "peptide": row["Peptide"],
        "protein": row["Protein"],
        "kw_stat": stat,
        "kw_pval": pval,
        "is_significant": pval < 0.05,
    })

kw_df = pd.DataFrame(kw_results)
kw_df.sort_values(by=["is_significant", "kw_pval", "peptide"], ascending=[False, True, True], inplace=True)


# * Using ANOVA for peptides normal in all 3 groups
anova_results = []
for peptide in normal_all_3:
    row = df.loc[df["Peptide"] == peptide].iloc[0]
    values_by_group = [row[groups[g]].astype(float).values for g in group_names]
    stat, pval = f_oneway(*values_by_group)
    anova_results.append({
        "peptide": row["Peptide"],
        "protein": row["Protein"],
        "anova_stat": stat,
        "anova_pval": pval,
        "is_significant": pval < 0.05,
    })
anova_df = pd.DataFrame(anova_results)
anova_df.sort_values(by=["is_significant", "anova_pval", "peptide"], ascending=[False, True, True], inplace=True)

print(f"Step 5: Statistical tests between groups done.")


### Step 6: Applying Bonferroni and Benjamini-Hochberg correction and detecting significant peptides


In [ ]:
sig_kw = set(kw_df.loc[kw_df["is_significant"], "peptide"])
sig_anova = set(anova_df.loc[anova_df["is_significant"], "peptide"])
sig_peptides = sorted(sig_kw.union(sig_anova))
n_sig = len(sig_peptides)

anova_pvals = anova_df["anova_pval"].values

_, anova_pvals_bonf, _, _ = multipletests(anova_pvals, method="bonferroni")
_, anova_pvals_bh, _, _    = multipletests(anova_pvals, method="fdr_bh")

anova_df["anova_pval_bonf"] = anova_pvals_bonf
anova_df["anova_pval_fdr_bh"] = anova_pvals_bh

anova_df["is_significant_bonf"] = anova_df["anova_pval_bonf"] < 0.05
anova_df["is_significant_fdr"]  = anova_df["anova_pval_fdr_bh"] < 0.05

sig_anova_fdr = set(anova_df.loc[anova_df["is_significant_fdr"], "peptide"])
sig_peptides_fdr = sorted(sig_anova_fdr)

sig_anova_bonf = set(anova_df.loc[anova_df["is_significant_bonf"], "peptide"])
sig_peptides_bonf = sorted(sig_anova_bonf)

if n_sig == 0:
    print(f"Step 7: No significant peptides found. Skipping visualizations.")

print(f"Step 6: unique significant peptides from Kruskal/ANOVA after Bonferroni correction:", len(sig_peptides))
print(f"Step 6: unique significant peptides from Kruskal/ANOVA after FDR correction:", len(sig_peptides_bonf))
print(f"Step 6: unique significant peptides from Kruskal/ANOVA:", n_sig)

if cfg.run.use_correction:
    sig_peptides = sig_peptides_fdr

### Step 7: Visualizations of the groups of peptides that are significant

**7.1:** Get distinct significant peptides from Kruskal and ANOVA


In [ ]:
group_colors = {
    "PD":  "#636EFA",   # blue
    "RBD": "#EF553B",   # red
    "CON": "#00CC96",   # green
}

group_names = list(groups.keys())

**7.2**: Box plots

In [ ]:
fig_box = make_subplots(
    rows=1,
    cols=len(sig_peptides),
    shared_yaxes=False,
    subplot_titles=sig_peptides
)
for col_idx, peptide in enumerate(sig_peptides, start=1):
    row_pep = df.loc[df["Peptide"] == peptide].iloc[0]
    for g in group_names:
        values = row_pep[groups[g]].astype(float).values
        fig_box.add_trace(
            go.Box(
                y=values,
                name=g,
                boxpoints="outliers",
                showlegend=(col_idx == 1),
                marker_color=group_colors.get(g),
            ),
            row=1,
            col=col_idx
        )
width_box = max(350 * len(sig_peptides), 800)
height_box = 400
fig_box.update_layout(
    title="Significant Peptides – Box Plots per Peptide (3 groups each)",
    width=width_box,
    height=height_box
)
fig_box.show()
print(f"Step 7.2: Box plots done.")

**7.3**: Histograms: median bars

In [ ]:
fig_hist = make_subplots(
    rows=1,
    cols=len(sig_peptides),
    shared_yaxes=False,
    subplot_titles=sig_peptides
)
for col_idx, peptide in enumerate(sig_peptides, start=1):
    row_pep = df.loc[df["Peptide"] == peptide].iloc[0]
    for g in group_names:
        values = row_pep[groups[g]].astype(float).values
        values = values[~np.isnan(values)]
        if len(values) == 0:
            median_val = np.nan
        else:
            median_val = float(np.median(values))
        fig_hist.add_trace(
            go.Bar(
                x=[g],
                y=[median_val],
                name=g,
                marker_color=group_colors.get(g),
                showlegend=(col_idx == 1)
            ),
            row=1,
            col=col_idx
        )
fig_hist.update_layout(
    barmode="group"
)
width_hist = max(200 * len(sig_peptides), 400)
height_hist = 350
fig_hist.update_layout(
    title="Significant Peptides – Group Medians per Peptide (3 bars each)",
    width=width_hist,
    height=height_hist
)
fig_hist.show()


**7.4**: Density distributions

In [ ]:
fig_dist = make_subplots(
    rows=1,
    cols=len(sig_peptides),
    shared_yaxes=False,
    subplot_titles= sig_peptides
)
for col_idx, peptide in enumerate( sig_peptides, start=1):
    row_pep = df.loc[df["Peptide"] == peptide].iloc[0]

    all_vals = []
    group_vals = {}
    for g in group_names:
        vals = row_pep[groups[g]].astype(float).values
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            continue
        group_vals[g] = vals
        all_vals.append(vals)
    if not all_vals:
        continue
    all_vals = np.concatenate(all_vals)
    x_min, x_max = float(all_vals.min()), float(all_vals.max())
    if x_min == x_max:
        x_min -= 1e-6
        x_max += 1e-6
    xs = np.linspace(x_min, x_max, 200)

    for g in group_names:
        if g not in group_vals:
            continue
        vals = group_vals[g]
        if len(vals) < 2:
            continue
        kde = gaussian_kde(vals)
        ys = kde(xs)
        fig_dist.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                name=g,
                showlegend=(col_idx == 1),
                line=dict(color=group_colors.get(g)),
            ),
            row=1,
            col=col_idx
        )
width_dist = max(350 * len(sig_peptides), 800)
height_dist = 350
fig_dist.update_layout(
    title="Significant Peptides – Density Distributions per Peptide (3 groups each)",
    width=width_dist,
    height=height_dist
)
fig_dist.show()
print(f"Step 7.4: Distribution plots saved.")

